# 🔍 SEC EDGAR Financial Data — Exploration Notebook
**Project:** SEC EDGAR Financial Ratio Analysis  
**Engineer:** Meet Saini  
**Last Updated:** 2026-05-28

---

### Notebook Objectives
1. Build two master Delta tables (`edgar_sub_all`, `edgar_num_all`)
2. Profile the dataset — row counts, columns, nulls
3. Discover financial tags relevant to ratio analysis
4. Analyze industry and sector distribution
5. Identify filing trends across Pre-COVID, COVID-Impact, Post-COVID periods

### Master Tables
- `edgar_sub_all` → All 28 quarters of filing metadata unified
- `edgar_num_all` → All 28 quarters of financial numbers unified

## Step 1 — Environment Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, lit
import logging

spark = SparkSession.builder.getOrCreate()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("edgar_exploration")

TARGET_YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

print(f"✅ Spark session ready | Version: {spark.version}")
print(f"✅ Target years: {TARGET_YEARS}")

## Step 2 — Build Master Tables

Unions all 28 quarters into two master Delta tables.  
Uses `unionByName` to handle minor schema differences across quarters.  
Skips missing tables and logs warnings.

In [0]:
def build_master_table(file_type: str, target_years: list) -> None:
    """
    Unions all quarterly Delta tables of a given type into one master table.
    
    Args:
        file_type: 'sub' or 'num'
        target_years: list of years to include
    """
    master_df = None
    skipped = []
    processed = []
    
    for year in target_years:
        for q in range(1, 5):
            table_name = f"edgar_{file_type}_{year}_q{q}"
            try:
                df = spark.table(table_name)
                if master_df is None:
                    master_df = df
                else:
                    master_df = master_df.unionByName(df, allowMissingColumns=True)
                processed.append(table_name)
                logger.info(f"✅ Added: {table_name}")
            except Exception as e:
                skipped.append(table_name)
                logger.warning(f"⚠️ Skipped: {table_name} — {e}")
    
    if master_df is None:
        logger.error("❌ No tables found — master table not created")
        return
    
    # Save as Delta table
    master_table_name = f"edgar_{file_type}_all"
    master_df.write.format("delta").mode("overwrite").saveAsTable(master_table_name)
    
    print(f"\n{'='*50}")
    print(f"✅ Master table created : {master_table_name}")
    print(f"✅ Tables processed     : {len(processed)}")
    print(f"⚠️ Tables skipped       : {len(skipped)}")
    if skipped:
        print(f"   Skipped list        : {skipped}")
    print(f"{'='*50}")


print("✅ build_master_table function defined")

### Step 2a — Build `edgar_sub_all`
> ⏱️ Estimated time: 2-3 minutes

In [0]:
build_master_table("sub", TARGET_YEARS)

### Step 2b — Build `edgar_num_all`
> ⏱️ Estimated time: 20-30 minutes (84M+ rows)

In [0]:
build_master_table("num", TARGET_YEARS)